In [1]:
%pip install -q xgboost pandas numpy scikit-learn matplotlib seaborn

In [2]:
from google.colab import files

files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"barathmr10","key":"5b54d9156f2317d8cd0ef8041f106434"}'}

In [3]:
!pip install -q kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!mkdir -p /content/paysim

!kaggle datasets download -d ealaxi/paysim1 -p /content/paysim --unzip

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/ealaxi/paysim1
License(s): CC-BY-SA-4.0
100% 178M/178M [00:01<00:00, 114MB/s] 



In [4]:
import os
import glob

files = glob.glob("/content/paysim/**/*", recursive=True)

for file in files:
    if os.path.isfile(file):
        print(file, f"({os.path.getsize(file) / (1024**2):.2f} MB)")

/content/paysim/PS_20174392719_1491204439457_log.csv (470.67 MB)


In [5]:
!find /content/paysim -type f -name "*.csv" -exec wc -l {} \;

6362621 /content/paysim/PS_20174392719_1491204439457_log.csv


In [6]:
import glob
import pandas as pd

csv_files = glob.glob("/content/paysim/**/*.csv", recursive=True)

print("CSV files found:", csv_files)

file_path = csv_files[0]
df = pd.read_csv(file_path)

print("\nDataset shape:", df.shape)
print("Step range:", df["step"].min(), "to", df["step"].max())
print("\nFraud distribution:")
print(df["isFraud"].value_counts(dropna=False))

CSV files found: ['/content/paysim/PS_20174392719_1491204439457_log.csv']

Dataset shape: (6362620, 11)
Step range: 1 to 743

Fraud distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [14]:

# Separate input features (X) and target (y)

target = "isFraud"

features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

X = df[features].copy()
y = df[target].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())


Feature shape: (6362620, 7)
Target shape: (6362620,)

Features:
['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

Target distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [19]:

from sklearn.model_selection import train_test_split

# 1. Reserve 5% as a stratified holdout
X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X,
    y,
    test_size=0.05,
    random_state=42,
    stratify=y
)

# 2. Split 95% development data into 80% train portion and 20% test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_dev,
    y_dev,
    test_size=0.20,
    random_state=42,
    stratify=y_dev
)

# 3. Split the train portion into 90% training and 10% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.10,
    random_state=42,
    stratify=y_train_full
)

# 4. Check all splits
for name, features, labels in [
    ("Train", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Test", X_test, y_test),
    ("Holdout", X_holdout, y_holdout)
]:
    print(
        f"{name}: {len(features):,} rows | "
        f"Fraud: {int(labels.sum()):,} | "
        f"Fraud rate: {labels.mean() * 100:.3f}%"
    )


Train: 4,352,031 rows | Fraud: 5,618 | Fraud rate: 0.129%
Validation: 483,560 rows | Fraud: 624 | Fraud rate: 0.129%
Test: 1,208,898 rows | Fraud: 1,560 | Fraud rate: 0.129%
Holdout: 318,131 rows | Fraud: 411 | Fraud rate: 0.129%


In [21]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# 1. Create and fit the encoder using training data only
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype=int
)

encoder.fit(X_train[["type"]])

# 2. Function to encode the type column
def encode_features(X):
    encoded_type = encoder.transform(X[["type"]])

    type_df = pd.DataFrame(
        encoded_type,
        columns=encoder.get_feature_names_out(["type"]),
        index=X.index
    )

    return pd.concat(
        [X.drop(columns=["type"]), type_df],
        axis=1
    )

# 3. Encode all four splits
X_train_encoded = encode_features(X_train)
X_val_encoded = encode_features(X_val)
X_test_encoded = encode_features(X_test)
X_holdout_encoded = encode_features(X_holdout)

# 4. Check the results
print("Train:", X_train_encoded.shape)
print("Validation:", X_val_encoded.shape)
print("Test:", X_test_encoded.shape)
print("Holdout:", X_holdout_encoded.shape)

print("\nEncoded columns:")
print(X_train_encoded.columns.tolist())

Train: (4352031, 11)
Validation: (483560, 11)
Test: (1208898, 11)
Holdout: (318131, 11)

Encoded columns:
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [22]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print("Scale pos weight:", scale_pos_weight)

Scale pos weight: 773.6584193663225


In [23]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    device="cuda",
    random_state=42
)

model.fit(
    X_train_encoded,
    y_train,
    eval_set=[(X_val_encoded, y_val)],
    verbose=20
)

[0]	validation_0-aucpr:0.54059
[20]	validation_0-aucpr:0.77474
[40]	validation_0-aucpr:0.89600
[60]	validation_0-aucpr:0.92301
[80]	validation_0-aucpr:0.93501
[100]	validation_0-aucpr:0.94146
[120]	validation_0-aucpr:0.94627
[140]	validation_0-aucpr:0.95205
[160]	validation_0-aucpr:0.95650
[180]	validation_0-aucpr:0.95924
[199]	validation_0-aucpr:0.96031


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [24]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

# Predict fraud probabilities
y_val_prob = model.predict_proba(X_val_encoded)[:, 1]

# Apply the default threshold of 0.5
y_val_pred = (y_val_prob >= 0.5).astype(int)

print("Average Precision:", average_precision_score(y_val, y_val_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred, digits=4))

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [17:04:38] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Average Precision: 0.9603358536268866

Confusion Matrix:
[[481742   1194]
 [     2    622]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9975    0.9988    482936
           1     0.3425    0.9968    0.5098       624

    accuracy                         0.9975    483560
   macro avg     0.6713    0.9972    0.7543    483560
weighted avg     0.9991    0.9975    0.9981    483560



In [25]:
from sklearn.metrics import precision_score, recall_score

thresholds = [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]

for threshold in thresholds:
    y_pred = (y_val_prob >= threshold).astype(int)

    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"FP: {fp} | FN: {fn}"
    )

Threshold: 0.50 | Precision: 0.3425 | Recall: 0.9968 | FP: 1194 | FN: 2
Threshold: 0.70 | Precision: 0.4402 | Recall: 0.9856 | FP: 782 | FN: 9
Threshold: 0.80 | Precision: 0.5092 | Recall: 0.9760 | FP: 587 | FN: 15
Threshold: 0.90 | Precision: 0.6330 | Recall: 0.9647 | FP: 349 | FN: 22
Threshold: 0.95 | Precision: 0.7571 | Recall: 0.9391 | FP: 188 | FN: 38
Threshold: 0.99 | Precision: 0.9477 | Recall: 0.8413 | FP: 29 | FN: 99


In [26]:
from sklearn.metrics import f1_score

for threshold in thresholds:
    y_pred = (y_val_prob >= threshold).astype(int)
    f1 = f1_score(y_val, y_pred)

    print(f"Threshold: {threshold:.2f} | F1-score: {f1:.4f}")

Threshold: 0.50 | F1-score: 0.5098
Threshold: 0.70 | F1-score: 0.6086
Threshold: 0.80 | F1-score: 0.6692
Threshold: 0.90 | F1-score: 0.7644
Threshold: 0.95 | F1-score: 0.8383
Threshold: 0.99 | F1-score: 0.8913


In [30]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    classification_report)

y_test_prob = model.predict_proba(X_test_encoded)[:, 1]

threshold = 0.90
y_test_pred = (y_test_prob >= threshold).astype(int)

print("Average Precision:",average_precision_score(y_test, y_test_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, digits=4))

Average Precision: 0.9599595332331137

Confusion Matrix:
[[1206399     939]
 [     55    1505]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9992    0.9996   1207338
           1     0.6158    0.9647    0.7517      1560

    accuracy                         0.9992   1208898
   macro avg     0.8079    0.9820    0.8757   1208898
weighted avg     0.9995    0.9992    0.9993   1208898



In [31]:
import pandas as pd

importance = pd.Series(
    model.feature_importances_,
    index=X_train_encoded.columns
).sort_values(ascending=False)

print(importance)

newbalanceOrig    0.625899
type_PAYMENT      0.195542
oldbalanceOrg     0.072098
amount            0.034693
type_TRANSFER     0.026391
newbalanceDest    0.019911
step              0.011113
type_CASH_OUT     0.008211
type_DEBIT        0.003035
oldbalanceDest    0.003006
type_CASH_IN      0.000101
dtype: float32


In [32]:
print(
    df.groupby("isFraud")["newbalanceOrig"]
      .describe()
)

             count           mean           std  min  25%  50%        75%  \
isFraud                                                                     
0        6354407.0  855970.228109  2.924987e+06  0.0  0.0  0.0  144730.74   
1           8213.0  192392.631836  1.965666e+06  0.0  0.0  0.0       0.00   

                 max  
isFraud               
0        43686616.33  
1        49585040.37  


In [33]:
print(
    df.groupby("isFraud")["newbalanceOrig"]
      .apply(lambda s: (s == 0).mean() * 100)
)

isFraud
0    56.677405
1    98.051869
Name: newbalanceOrig, dtype: float64


In [34]:
features_without_balance = [
    col for col in X_train_encoded.columns
    if col != "newbalanceOrig"
]

print(features_without_balance)

['step', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'newbalanceDest', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [35]:
from xgboost import XGBClassifier

model_without_balance = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    reg_lambda=1,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    device="cuda",
    random_state=42
)

model_without_balance.fit(
    X_train_encoded[features_without_balance],
    y_train,
    eval_set=[
        (X_val_encoded[features_without_balance], y_val)
    ],
    verbose=20
)

[0]	validation_0-aucpr:0.25079
[20]	validation_0-aucpr:0.72963
[40]	validation_0-aucpr:0.84002
[60]	validation_0-aucpr:0.85591
[80]	validation_0-aucpr:0.86575
[100]	validation_0-aucpr:0.87188
[120]	validation_0-aucpr:0.87681
[140]	validation_0-aucpr:0.88372
[160]	validation_0-aucpr:0.89079
[180]	validation_0-aucpr:0.89748
[199]	validation_0-aucpr:0.90295


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [36]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    classification_report
)

y_val_prob_without_balance = model_without_balance.predict_proba(
    X_val_encoded[features_without_balance]
)[:, 1]

threshold = 0.95
y_val_pred_without_balance = (
    y_val_prob_without_balance >= threshold
).astype(int)

print(
    "Average Precision:",
    average_precision_score(y_val, y_val_prob_without_balance)
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_without_balance))

print("\nClassification Report:")
print(classification_report(
    y_val, y_val_pred_without_balance, digits=4
))

Average Precision: 0.9029882783862015

Confusion Matrix:
[[482682    254]
 [    77    547]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9998    0.9995    0.9997    482936
           1     0.6829    0.8766    0.7677       624

    accuracy                         0.9993    483560
   macro avg     0.8414    0.9380    0.8837    483560
weighted avg     0.9994    0.9993    0.9994    483560



In [37]:
importance_without_balance = pd.Series(
    model_without_balance.feature_importances_,
    index=features_without_balance
).sort_values(ascending=False)

print(importance_without_balance)

type_CASH_IN      0.595950
type_PAYMENT      0.176410
oldbalanceOrg     0.061464
amount            0.054386
type_TRANSFER     0.042603
newbalanceDest    0.023918
type_CASH_OUT     0.017504
step              0.013452
type_DEBIT        0.009980
oldbalanceDest    0.004335
dtype: float32


In [39]:
import joblib

# Save the trained XGBoost model
model.save_model("fraud_model.json")

# Save the fitted OneHotEncoder
joblib.dump(encoder, "transaction_encoder.joblib")

print("Model and encoder saved successfully!")

Model and encoder saved successfully!


In [38]:
!pip install -q google-cloud-storage

In [43]:
from google.colab import files

files.download("fraud_model.json")
files.download("transaction_encoder.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [44]:
import pandas as pd
from google.colab import files

# Combine holdout features and actual labels
holdout_df = X_holdout.copy()
holdout_df["isFraud"] = y_holdout

# Save as CSV
holdout_df.to_csv("holdout_transactions.csv", index=False)

print("Holdout data saved successfully!")
print("Shape:", holdout_df.shape)

files.download("holdout_transactions.csv")

Holdout data saved successfully!
Shape: (318131, 8)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>